<a href="https://colab.research.google.com/github/galipaydin/pes3850/blob/main/notebooks/lab07_athlete_phenotypes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# PES 3850 · Lab 7 — Athlete phenotypes: clustering and PCA
**Artificial Intelligence Applications in Sport · Week 7**

⏱️ About 75 minutes · 🧪 This notebook creates its own data — nothing can fail to download

> **First:** `File → Save a copy in Drive`. Cells marked ✏️ **EDIT ME** are the ones you change.


### What you will produce
A set of named athlete profiles from an 18-test battery, defended with a validation the clustering never saw — and
proof, in one cell, that the same method produces beautiful groups from pure noise.

### Learning without labels
Most applied sport data has no outcome column. You have 250 athletes and 18 tests and a coach asking *what kinds
of athlete do we have?* That is unsupervised learning: find the structure that is already there — if any is.


---
## Block 1 — Setup


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
RNG = np.random.default_rng(7)
print('Setup complete.')


---
## Block 2 — The test battery

250 athletes, 18 tests. Two variables are held back — `match_minutes` and `position` — and are **never used in
the clustering**. They are how we will find out at the end whether the clusters mean anything.


In [ ]:
N   = 250
SEP = 1.3    # ✏️ EDIT ME (advanced) — how distinct the underlying athlete types really are.
             #    Set it to 0 and the groups vanish. Watch what k-means does then.

# Three genuine athlete archetypes, each a different mix of three latent qualities.
# Real squads are like this: types exist, but they overlap heavily.
grp = RNG.choice([0, 1, 2], N, p=[.38, .34, .28])
MU  = np.array([[ SEP,     -.5*SEP, -.2*SEP],      # type 0
                [-.6*SEP,   SEP,     .0    ],      # type 1
                [-.2*SEP,  -.4*SEP,  SEP   ]])     # type 2
latent = MU[grp] + RNG.normal(0, .55, (N, 3))
power, endurance, robust = latent[:, 0], latent[:, 1], latent[:, 2]

def t(base, w_p=0, w_e=0, w_r=0, noise=.45, dec=1):
    return np.round(base + w_p*power + w_e*endurance + w_r*robust + RNG.normal(0, noise, N), dec)

b = pd.DataFrame({
    'cmj_cm'            : t(41,   w_p=4.2,             noise=1.6),
    'sj_cm'             : t(37,   w_p=3.8,             noise=1.6),
    'drop_jump_rsi'     : t(1.6,  w_p=.28,             noise=.12, dec=2),
    'sprint10_s'        : t(1.74, w_p=-.07,            noise=.030, dec=3),
    'sprint30_s'        : t(4.18, w_p=-.16, w_e=-.04,  noise=.070, dec=3),
    'flying20_s'        : t(2.42, w_p=-.10,            noise=.040, dec=3),
    'cod_505_s'         : t(2.48, w_p=-.09, w_r=-.03,  noise=.050, dec=3),
    'yoyo_m'            : t(880,  w_e=210,             noise=90,  dec=0),
    'vo2max'            : t(56,   w_e=5.4,             noise=2.2),
    'vift_kmh'          : t(18.5, w_e=1.2,             noise=.50),
    'hr_rec60_bpm'      : t(31,   w_e=6.0,             noise=2.5, dec=0),
    'lactate_4mmol_kmh' : t(13.2, w_e=1.1,             noise=.50),
    'iso_hamstring_N'   : t(300,  w_r=52,  w_p=18,     noise=22,  dec=0),
    'iso_quad_N'        : t(420,  w_r=58,  w_p=24,     noise=26,  dec=0),
    'hip_add_abd_ratio' : t(1.02, w_r=.09,             noise=.040, dec=2),
    'sit_and_reach_cm'  : t(18,   w_r=4.5,             noise=2.0),
    'ybt_composite'     : t(94,   w_r=6.2,             noise=2.6),
    'plank_hold_s'      : t(112,  w_r=26,  w_e=12,     noise=12,  dec=0),
})

# held back — never given to the clustering
held = pd.DataFrame({
    'match_minutes': np.round(np.clip(1400 + 260*endurance + 190*robust + RNG.normal(0,240,N), 0, None)).astype(int),
    'position'     : np.where(power > .6, 'FW', np.where(endurance > .6, 'MF',
                              np.where(robust > .3, 'DF', 'SUB'))),
})

TESTS = list(b.columns)
print(f'{N} athletes × {len(TESTS)} tests')
print('(There really are three types in this squad — but nothing tells the algorithm that,')
print(" and it is your job to work out whether you could have known.)")
b.head()


---
## Block 3 — Standardise, then reduce

The tests are in different units — seconds, centimetres, newtons, metres. Without standardising, `yoyo_m` (which
runs to 1,300) would dominate `drop_jump_rsi` (which runs to 2) purely because of its scale.

**PCA** then asks: eighteen tests, but how many *qualities* are really being measured?


In [ ]:
Z = StandardScaler().fit_transform(b)

pca = PCA().fit(Z)
ev  = pca.explained_variance_ratio_

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].bar(range(1, len(ev)+1), ev*100, color='#1B6470')
ax[0].set_xlabel('Component'); ax[0].set_ylabel('% of variance'); ax[0].set_title('Scree plot')
ax[1].plot(range(1, len(ev)+1), np.cumsum(ev)*100, 'o-', color='#8A1C2B')
ax[1].axhline(80, ls='--', color='#999'); ax[1].set_xlabel('Components')
ax[1].set_ylabel('Cumulative %'); ax[1].set_title('How many qualities?')
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f'The first 3 components carry {np.cumsum(ev)[2]*100:.0f}% of everything the 18 tests measure.')


### Name the components

A component is just a weighted recipe of the original tests. Read the largest weights and give it a name a coach
would use — that translation is the entire skill.


In [ ]:
load = pd.DataFrame(pca.components_[:3].T, index=TESTS, columns=['PC1','PC2','PC3'])
for pc in ['PC1','PC2','PC3']:
    top = load[pc].abs().sort_values(ascending=False).head(5).index
    print(f'\n{pc}  ({ev[["PC1","PC2","PC3"].index(pc)]*100:.0f}% of variance)')
    print(load.loc[top, pc].round(2).to_string())

print('\n✏️ Write a name for each in the interpretation cell — e.g. "explosive power", "aerobic capacity".')


---
## Block 4 — How many groups?

k-means needs you to state how many clusters to find. It will always find exactly that many, whether or not any
exist. Two diagnostics help you choose, and neither of them decides for you.


In [ ]:
ks, inertias, sils = range(2, 9), [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(Z, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ks, inertias, 'o-', color='#1B6470'); ax[0].set_xlabel('k'); ax[0].set_ylabel('Inertia')
ax[0].set_title('Elbow — look for the bend')
ax[1].plot(ks, sils, 'o-', color='#8A1C2B'); ax[1].set_xlabel('k'); ax[1].set_ylabel('Silhouette')
ax[1].set_title('Silhouette — higher is tighter')
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

print('Best silhouette at k =', ks[int(np.argmax(sils))], f'(score {max(sils):.3f})')
print('A silhouette below about 0.25 usually means there are no real groups here.')


---
## ✏️ EDIT ME — choose k and profile the clusters


In [ ]:
K = 3   # ✏️ EDIT ME — your choice, defended in the interpretation cell

km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(Z)
b['cluster'] = km.labels_

profile = b.groupby('cluster')[TESTS].mean()
z_profile = (profile - b[TESTS].mean()) / b[TESTS].std()   # in SD units, so tests are comparable

fig, ax = plt.subplots(figsize=(11, 4.2))
im = ax.imshow(z_profile.values, cmap='RdBu_r', vmin=-1.5, vmax=1.5, aspect='auto')
ax.set_xticks(range(len(TESTS))); ax.set_xticklabels(TESTS, rotation=55, ha='right', fontsize=8)
ax.set_yticks(range(K)); ax.set_yticklabels([f'Cluster {i}  (n={sum(km.labels_==i)})' for i in range(K)])
ax.set_title('Cluster profiles, in standard deviations from the squad mean')
plt.colorbar(im, label='SD from mean'); plt.tight_layout(); plt.show()

print(z_profile.round(2).T.to_string())


---
## Block 5 — The test that actually matters

Anyone can produce clusters. The question is whether they correspond to anything real.

`match_minutes` and `position` were **never given to the algorithm**. If the clusters differ on them, the structure
is real. If they do not, you have partitioned noise.


In [ ]:
chk = b[['cluster']].join(held)
print('Mean match minutes by cluster:')
print(chk.groupby('cluster')['match_minutes'].agg(['mean','std','count']).round(0).to_string())
print('\nPosition composition (%):')
print((pd.crosstab(chk.cluster, chk.position, normalize='index')*100).round(0).to_string())

spread = chk.groupby('cluster')['match_minutes'].mean()
print(f'\nSpread between clusters: {spread.max()-spread.min():.0f} minutes '
      f'(squad SD = {chk.match_minutes.std():.0f}).')
print('If that spread is small relative to the SD, your clusters are decorative.')


---
## Block 6 — Break it 💥

The same pipeline, run on **pure random numbers** with the same shape. Watch it produce clean, balanced,
convincing clusters out of nothing at all.


In [ ]:
noise = RNG.normal(0, 1, (N, len(TESTS)))
km_n  = KMeans(n_clusters=K, n_init=10, random_state=0).fit(noise)
p_n   = PCA(n_components=2).fit_transform(noise)
p_r   = PCA(n_components=2).fit_transform(Z)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].scatter(p_r[:,0], p_r[:,1], c=km.labels_,    cmap='Set2', s=26)
ax[0].set_title(f'Real test battery  ·  silhouette {silhouette_score(Z, km.labels_):.3f}')
ax[1].scatter(p_n[:,0], p_n[:,1], c=km_n.labels_,  cmap='Set2', s=26)
ax[1].set_title(f'Pure random noise  ·  silhouette {silhouette_score(noise, km_n.labels_):.3f}')
for a in ax: a.set_xlabel('PC1'); a.set_ylabel('PC2'); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

print('Both pictures look like clusters. Only one of them is.')
print('This is why the held-out validation in Block 5 is not optional.')


---
## Interpretation ✍️ **(graded)**

About 200 words, written as if for a coach:

1. Name PC1, PC2 and PC3 in sport-science language.
2. State the `K` you chose and **defend it** using the elbow, the silhouette and the profiles — not just one of them.
3. Give each cluster a name a coach would use, and one training implication for each.
4. Does the held-out validation support your clusters? Say so plainly if it does not.

---

**Your answer:**

*(replace this line)*


---
## AI Use Log

Complete honestly — it is part of the mark. Write *none* if you used no AI tool.

| Tool | What I asked it | What it gave me | What I changed or rejected |
|---|---|---|---|
|  |  |  |  |


---
### Submitting

1. `File → Save a copy in Drive` · 2. `Share → Anyone with the link → Viewer` · 3. Paste the link into ODTÜClass.
